# 02 — Feature Engineering
Builds the modelling dataset: aggregates `transactions` to customer grain,
joins customers + credit_history + loan_applications, engineers ratio
features, and applies leakage control. Logic lives in `src/features.py`
so the notebook and `src/train.py` share one source of truth.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
from features import build_feature_table, get_model_ready, LEAKAGE_COLUMNS, CATEGORICAL_COLUMNS

pd.set_option('display.max_columns', 50)
DB = '../data/credit_risk.db'

## 1. Build the joined, feature-engineered table

In [ ]:
features = build_feature_table(DB)
print(f"Feature table: {features.shape[0]:,} rows x {features.shape[1]} columns")
features.head(3)

## 2. Confirm leakage columns were dropped
Approval_status, approved_amount, decision_date and disbursed_flag are only known **after** the underwriting decision -- they must never enter the model.

In [ ]:
assert not any(col in features.columns for col in LEAKAGE_COLUMNS), "Leakage column leaked into features!"
print("Leakage check passed:", LEAKAGE_COLUMNS, "correctly excluded")

## 3. Restrict to the disbursed, outcome-observed population and encode

In [ ]:
model_df = get_model_ready(features)
print(f"Model-ready rows: {model_df.shape[0]:,} (disbursed loans with an observed outcome)")
print(f"Model-ready columns: {model_df.shape[1]} (after one-hot encoding)")
print(f"Default rate in modelling population: {model_df['loan_default'].mean():.1%}")
model_df.head(3)

## 4. Sanity-check engineered ratios

In [ ]:
features[['debt_to_limit_ratio', 'loan_to_income_ratio', 'txn_inflow_outflow_ratio']].describe()